In [1]:
# !pip install mysql-connector-python 
# !pip install sqlalchemy

In [1]:
# load library
import mysql.connector
from mysql.connector import Error
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError
from datetime import datetime
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

### Connexion Engine

In [2]:
# Function : Établit une connexion à une base de données MySQL
# paramètre : nom de la base de données
# return : moteur (engine) SQLAlchemy de la base de données
def connect_engine_db(nom_db):
    try:
        # Création de l'URL de connexion pour le socket Unix
        url = f"mysql+mysqlconnector://root:NgocLam2301@/{nom_db}?unix_socket=/tmp/mysql.sock"
        # Initialisation de l'engine
        engine = create_engine(url)

        # Vérifier si la connexion fonctionne réellement, 
        with engine.connect() as connection:
            connection.execute(text("SELECT 1"))
            
        print(f"MOTEUR PRÊT ET VÉRIFIÉ : {nom_db}")
        return engine
    
    except Error as e:
        # Affichage de l'erreur si la connexion échoue
        print(f"Erreur lors de la création du moteur pour {nom_db}: {e}")
        return None

In [3]:
# Function : Exécute une requête SQL et retourne un DataFrame Pandas.
# paramètre : - query pour lire la données dans la table
#             - engine de la base de données
# return : données de table sous forme dataframe 
def sql_df(query, engine):
    return pd.read_sql_query(query, engine)

### File source

In [4]:
# Function : convertir fichier CSV et retourner un DataFrame Pandas, afficher des informations necessaires
# paramètre : - lien de fichier
# return : données de table sous forme dataframe 
def csv_df(url, afficher = True):
    # read et convertir fichier CSV vers dataframe
    df = pd.read_csv(url, low_memory=False)
    
    # surprimer ligne contient le nom de colonnne
    df_source_brut = df[df["code_nature_culture"] != 'code_nature_culture']

    # Đồng nhất tất cả null values thành NaN
    df_source_brut = df_source_brut.fillna(np.nan)

    if afficher:
        print(df_source_brut["code_type_local"].unique())
        print(f"Nombre de lignes : {df_source_brut.shape[0]}")
        print(f"Nombre de colonnes : {df_source_brut.shape[1]}")
        display(df_source_brut.head().T)

    return df_source_brut

# CSV to Source

In [5]:
# Function : Creation une table qui contient le nom de table dans une base de donnees (et ses colonnes si demande)
# paramètre : - engine de la base de donnees
#             - retourner par de fault les colonnes de la table 
# return : données de table sous forme dataframe 
def sql_df_db_tab(engine_db, colonne = True):
    list_table = sql_df("show tables", engine_db)
    list_table.columns.array[0] = 'table_name'
    # list_table.rename(columns={"Tables_in_db_source":"table_name"}, inplace=True)

    # si il demande des colonnes de la table
    if colonne:
        # create colonne vide
        list_table["colonnes"] = ""
        
        for tab in list_table["table_name"]:
            query = f"select * from {tab}"
            result = sql_df(query, engine_db)
            noms_colonnes = ", ".join(result.columns)
            
            list_table.loc[list_table["table_name"] == tab, ["colonnes"]] = noms_colonnes
    
    display(list_table)
    return list_table

In [6]:
def tab_priority(tab_target, surfix = None):
    # Creation Dataframe a partir de l'odre pour eviter l'erreur de cle etranger
    # order_insert = ["Commune", "Local", "Culture", "Culture_speciale", "Mutation", "Parcelle", "Lot", "Adresse"]
    order_insert = ["Commune", "Local", "Culture", "Culture_speciale", "Mutation", "Parcelle", "Adresse", "Lot"]
    
    if surfix:
        order_insert = [f"{surfix}_{col}" for col in order_insert]
        
    df_priority = pd.DataFrame({
        "table_name": order_insert,
        "priority": range(len(order_insert)) 
    })
    
    # jointure avec le table qui contient des colonnes
    table_ordered = pd.merge(df_priority, tab_target, on="table_name", how="left")
    display(table_ordered)
    return table_ordered

def truncate_tab(engine_db, table):
    with engine_db.connect() as conn:
        # turn off la verification la cle etranger
        conn.execute(text("SET FOREIGN_KEY_CHECKS = 0;"))
        conn.execute(text(f"TRUNCATE TABLE {table}"))
        # turn on la verification la cle etranger
        conn.execute(text("SET FOREIGN_KEY_CHECKS = 1;"))
        # commmit pour sauvegarder
        conn.commit()
        
def sql_df_db_source(list_tab, df, engine_db):
    # jointure avec le table qui contient des colonnes dans l'ordre
    list_table_ordered = tab_priority(list_tab)
    
    # extraire des colonne pertient pour chaque colonne et charger dans la base de données
    for index, row in list_table_ordered.iterrows():
        # # list de colonne
        # col = list_tab.at[index, 'colonnes']
        # # nom de table
        # tab = list_tab.at[index, 'table_name']
        # liste_colonnes = col.split(", ")
        # print(tab)
        
        # # surprimer des colonnee qui est auto-increment et cette colonne n'existe pas dans le fichier CSV
        # if tab == "Lot":
        #     liste_colonnes.remove("id_lot")
        # elif tab == "Adresse":
        #     liste_colonnes.remove("id_adresse")
       
        # # extraire les colonnnes pertinentes
        # df_test = df[liste_colonnes].copy()  
        # # surprimer les lignes que leurs attributs sont nulles
        # df_test.dropna(how="all", inplace=True, ignore_index=True)
        # # surprimer des duplications
        # df_test.drop_duplicates(inplace=True, ignore_index=True)
    
        # # pour les table Mutation et Parcelle, on doit surprimer la duplication sur la cle primaire
        # if tab == "Mutation":
        #     df_test = df_test.drop_duplicates(subset=['id_mutation'])
        # elif tab == "Parcelle":
        #     df_test = df_test.drop_duplicates(subset=['id_parcelle'])
    
        # # truncate les donnees avant inserer des donnees
        # truncate_tab(engine_db, tab)
    
        # # changer ver sql et charger des donnees dans la base de donnees
        # df_test.to_sql(name=tab, con=engine_db, if_exists='append', index=False)

        # list de colonne
        col = list_table_ordered.at[index, 'colonnes']
        # nom de table
        tab = list_table_ordered.at[index, 'table_name']
        liste_colonnes = col.split(", ")

        # surprimer des colonnee qui est auto-increment et cette colonne n'existe pas dans le fichier CSV
        if tab == "Lot":
            liste_colonnes.remove("id_lot")
            liste_colonnes.remove("id_adresse")
        elif tab == "Adresse":
            liste_colonnes.remove("id_adresse")

        # extraire les colonnnes pertinentes
        df_test = pd.DataFrame()
        if tab == "Lot":
            # col_sub = list_table_ordered.loc[list_table_ordered['table_name'] == 'Adresse', 'colonnes'].split(',')
            col_sub = list_table_ordered.at[index - 1, 'colonnes'].split(", ")
            col_sub.remove("id_adresse")
            list_sub = liste_colonnes + col_sub            
            df_test = df[list_sub].copy()
            df_adresse_check = sql_df("select * from Adresse", engine_db)
            df_test = df_test.merge(df_adresse_check, how='left', on=col_sub)
            df_test = df_test.drop(columns=col_sub)
            display(df_test.head())
        else:
            df_test = df[liste_colonnes].copy()
            
        # # extraire les colonnnes pertinentes  
        # df_test = df[liste_colonnes].copy()
        
        # surprimer les lignes que leurs attributs sont nulles
        df_test.dropna(how="all", inplace=True, ignore_index=True)
        # surprimer des duplications
        df_test.drop_duplicates(inplace=True, ignore_index=True)
    
        # pour les table Mutation et Parcelle, on doit surprimer la duplication sur la cle primaire
        if tab == "Mutation":
            df_test = df_test.drop_duplicates(subset=['id_mutation'])
        elif tab == "Parcelle":
            df_test = df_test.drop_duplicates(subset=['id_parcelle'])
    
        # truncate les donnees avant inserer des donnees
        truncate_tab(engine_db, tab)
    
        # changer ver sql et charger des donnees dans la base de donnees
        df_test.to_sql(name=tab, con=engine_db, if_exists='append', index=False)

# df_csv = csv_df(url_csv)
# test_tab = sql_df_db_tab(engine_db_source)
# sql_df_db_source(test_tab, df_csv, engine_db_source)

In [7]:
# Verifier les contenus
def verifier_db_tab(list_tab, engine_db):
    for tab in list_tab.iloc[:,0]:
        query = f"select * from {tab}"
        result = sql_df(query, engine_db)
        print(result.shape)
        display(result.head())

In [8]:
def flux_csv_source(url_csv = "./../../dvf_test.csv"): 
    # creation l'engine de "db_source"
    engine_db_source = connect_engine_db('db_source') 
    
    # creation le dataframe
    df_csv = csv_df(url_csv, afficher=True)
    
    # creation la dataframe qui contient le nom de table et ses colonnes
    tab_db_source = sql_df_db_tab(engine_db_source)
    
    # charger dataframe dans la base de donnees
    sql_df_db_source(tab_db_source, df_csv, engine_db_source)
    # Verifier les contenus
    verifier_db_tab(tab_db_source, engine_db_source)
    
    # fermer les connexions
    engine_db_source.dispose()

In [9]:
flux_csv_source()

MOTEUR PRÊT ET VÉRIFIÉ : db_source
[nan '1' '3' '2' '4']
Nombre de lignes : 174863
Nombre de colonnes : 41


,0,1,2,3,4
id_mutation,2022-364781,2022-374264,2022-374264,2022-364781,2022-364786
date_mutation,2022-01-18,2022-09-21,2022-09-21,2022-01-18,2022-01-28
numero_disposition,1,1,1,1,2
nature_mutation,Vente,Vente,Vente,Vente,Vente
valeur_fonciere,245000.00,185000.00,185000.00,245000.00,4238450.00
adresse_numero,NaN,NaN,NaN,NaN,NaN
adresse_suffixe,NaN,NaN,NaN,NaN,NaN
adresse_nom_voie,LE HAMEL,LE SCABRE,LE SCABRE,LE HAMEL,LA MAISON ROUGE
adresse_code_voie,B038,B046,B046,B038,B026
code_postal,27400,27730,27730,27400,27460


,table_name,colonnes
0,Adresse,"id_adresse, adresse_numero, adresse_suffixe, a..."
1,Commune,"code_commune, nom_commune, ancien_code_commune..."
2,Culture,"code_nature_culture, nature_culture"
3,Culture_speciale,"code_nature_culture_speciale, nature_culture_s..."
4,Local,"code_type_local, type_local"
5,Lot,"id_lot, lot1_numero, lot1_surface_carrez, lot2..."
6,Mutation,"id_mutation, nature_mutation, date_mutation, v..."
7,Parcelle,"id_parcelle, ancien_id_parcelle, numero_dispos..."


,table_name,priority,colonnes
0,Commune,0,"code_commune, nom_commune, ancien_code_commune..."
1,Local,1,"code_type_local, type_local"
2,Culture,2,"code_nature_culture, nature_culture"
3,Culture_speciale,3,"code_nature_culture_speciale, nature_culture_s..."
4,Mutation,4,"id_mutation, nature_mutation, date_mutation, v..."
5,Parcelle,5,"id_parcelle, ancien_id_parcelle, numero_dispos..."
6,Adresse,6,"id_adresse, adresse_numero, adresse_suffixe, a..."
7,Lot,7,"id_lot, lot1_numero, lot1_surface_carrez, lot2..."


,lot1_numero,lot1_surface_carrez,lot2_numero,lot2_surface_carrez,lot3_numero,lot3_surface_carrez,lot4_numero,lot4_surface_carrez,lot5_numero,lot5_surface_carrez,nombre_pieces_principales,surface_reelle_bati,surface_terrain,numero_volume,nombre_lots,id_parcelle,code_nature_culture,code_nature_culture_speciale,code_type_local,id_adresse
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,102.00,NaN,0,273390000D0554,S,NaN,NaN,1
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,500.00,NaN,0,27119000ZB0134,S,NaN,NaN,2
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1520.00,NaN,0,27119000ZB0135,S,NaN,NaN,2
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1558.00,NaN,0,27339000ZC0028,P,NaN,NaN,1
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,280.00,NaN,0,270080000C1501,S,NaN,NaN,3


(44256, 7)


... (11 outputs left out)

(47916, 4)


,id_mutation,nature_mutation,date_mutation,valeur_fonciere
0,2021-370814,Vente,2021-02-26,145000.0
1,2021-372118,Vente,2021-04-07,30000.0
2,2021-372840,Vente,2021-05-07,20000.0
3,2021-372841,Vente,2021-05-06,2320.0
4,2021-372842,Vente,2021-05-05,30000.0


(75778, 6)


,id_parcelle,ancien_id_parcelle,numero_disposition,longitude,latitude,id_mutation
0,270010000A0016,None,1,0.715072,49.168240,2024-279960
1,270010000A0061,None,1,0.710988,49.165466,2023-306894
2,270010000A0078,None,1,0.711032,49.161631,2025-298261
3,270010000A0079,None,1,0.710678,49.161290,2025-298261
4,270010000A0094,None,1,0.708208,49.161878,2022-367198


# Source to DSA

In [10]:
# def flux_source_dsa_ods(engine_source, engine_target, mode="Source_DSA"):
#     # creation l'engine source
#     engine_source = connect_engine_db(engine_source) 
#     # creation l'engine targer
#     engine_target = connect_engine_db(engine_target)

#     # creation la dataframe qui contient le nom de table et ses colonnes de base de donnees source
#     tab_source = sql_df_db_tab(engine_source)
#     # creation la dataframe qui contient le nom de table et ses colonnes de base de donnees target
#     tab_taget = sql_df_db_tab(engine_target, False)

#     # jointure avec le table qui contient des colonnes dans l'ordre 
#     if mode != "Source_DSA":
#         list_table_ordered = tab_priority(tab_source, surfix='DSA')
#     else:
#         list_table_ordered = tab_priority(tab_source)
        
#     for tab in list_table_ordered["table_name"]:
#         query = f"select * from {tab}"
#         result = sql_df(query, engine_source)

#         if mode == "Source_DSA":
#             # truncate les donnees avant inserer des donnees
#             truncate_tab(engine_target, f"DSA_{tab}")      
        
#             # changer ver sql et charger des donnees dans la base de donnees
#             result.to_sql(name=f"DSA_{tab}", con=engine_target, if_exists='append', index=False)
#         else:
#             # changer ver sql et charger des donnees dans la base de donnees
#             result.to_sql(name=tab.replace("DSA", "ODS"), con=engine_target, if_exists='append', index=False)

#     verifier_db_tab(tab_taget, engine_target)

#     # fermer les connexions
#     engine_source.dispose()
#     engine_target.dispose()

def flux_source_dsa():
    # creation l'engine source
    engine_source = connect_engine_db('db_source') 
    # creation l'engine targer
    engine_target = connect_engine_db('db_dsa')

    # creation la dataframe qui contient le nom de table et ses colonnes de base de donnees source
    tab_source = sql_df_db_tab(engine_source)
    # creation la dataframe qui contient le nom de table et ses colonnes de base de donnees target
    tab_taget = sql_df_db_tab(engine_target, False)

    # jointure avec le table qui contient des colonnes dans l'ordre 
    list_table_ordered = tab_priority(tab_source)
        
    for tab in list_table_ordered["table_name"]:
        query = f"select * from {tab}"
        result = sql_df(query, engine_source)

        # truncate les donnees avant inserer des donnees
        truncate_tab(engine_target, f"DSA_{tab}")      
    
        # changer ver sql et charger des donnees dans la base de donnees
        result.to_sql(name=f"DSA_{tab}", con=engine_target, if_exists='append', index=False)
        
    verifier_db_tab(tab_taget, engine_target)

    # fermer les connexions
    engine_source.dispose()
    engine_target.dispose()

In [11]:
# flux_source_dsa_ods('db_source', 'db_dsa', mode="Source_DSA")
flux_source_dsa()

MOTEUR PRÊT ET VÉRIFIÉ : db_source
MOTEUR PRÊT ET VÉRIFIÉ : db_dsa


,table_name,colonnes
0,Adresse,"id_adresse, adresse_numero, adresse_suffixe, a..."
1,Commune,"code_commune, nom_commune, ancien_code_commune..."
2,Culture,"code_nature_culture, nature_culture"
3,Culture_speciale,"code_nature_culture_speciale, nature_culture_s..."
4,Local,"code_type_local, type_local"
5,Lot,"id_lot, lot1_numero, lot1_surface_carrez, lot2..."
6,Mutation,"id_mutation, nature_mutation, date_mutation, v..."
7,Parcelle,"id_parcelle, ancien_id_parcelle, numero_dispos..."


,table_name
0,DSA_Adresse
1,DSA_Commune
2,DSA_Culture
3,DSA_Culture_speciale
4,DSA_Local
5,DSA_Lot
6,DSA_Mutation
7,DSA_Parcelle


,table_name,priority,colonnes
0,Commune,0,"code_commune, nom_commune, ancien_code_commune..."
1,Local,1,"code_type_local, type_local"
2,Culture,2,"code_nature_culture, nature_culture"
3,Culture_speciale,3,"code_nature_culture_speciale, nature_culture_s..."
4,Mutation,4,"id_mutation, nature_mutation, date_mutation, v..."
5,Parcelle,5,"id_parcelle, ancien_id_parcelle, numero_dispos..."
6,Adresse,6,"id_adresse, adresse_numero, adresse_suffixe, a..."
7,Lot,7,"id_lot, lot1_numero, lot1_surface_carrez, lot2..."


(44256, 7)


,id_adresse,adresse_numero,adresse_suffixe,adresse_code_voie,adresse_nom_voie,code_postal,code_commune
0,1,None,None,B038,LE HAMEL,27400,27339
1,2,None,None,B046,LE SCABRE,27730,27119
2,3,None,None,B026,LA MAISON ROUGE,27460,27008
3,4,None,None,B030,LE VILLAGE,27500,27620
4,5,None,None,B033,ROUVILLE,27460,27008


... (10 outputs left out)

(47916, 4)


,id_mutation,nature_mutation,date_mutation,valeur_fonciere
0,2021-370814,Vente,2021-02-26,145000.0
1,2021-372118,Vente,2021-04-07,30000.0
2,2021-372840,Vente,2021-05-07,20000.0
3,2021-372841,Vente,2021-05-06,2320.0
4,2021-372842,Vente,2021-05-05,30000.0


(75778, 6)


,id_parcelle,ancien_id_parcelle,numero_disposition,longitude,latitude,id_mutation
0,270010000A0016,None,1,0.715072,49.168240,2024-279960
1,270010000A0061,None,1,0.710988,49.165466,2023-306894
2,270010000A0078,None,1,0.711032,49.161631,2025-298261
3,270010000A0079,None,1,0.710678,49.161290,2025-298261
4,270010000A0094,None,1,0.708208,49.161878,2022-367198


# DSA to ODS

In [12]:
# flux_source_dsa_ods('db_dsa', 'db_ods', mode="DSA_ODS")
def flux_dsa_ods():
    # creation l'engine source
    engine_source = connect_engine_db('db_dsa') 
    # creation l'engine targer
    engine_target = connect_engine_db('db_ods')

    # creation la dataframe qui contient le nom de table et ses colonnes de base de donnees source
    tab_source = sql_df_db_tab(engine_source)
    # creation la dataframe qui contient le nom de table et ses colonnes de base de donnees target
    tab_taget = sql_df_db_tab(engine_target, False)

    # jointure avec le table qui contient des colonnes dans l'ordre 
    list_table_ordered = tab_priority(tab_source)
        
    for tab in list_table_ordered["table_name"]:
        # lire table dans DSA
        result_dsa = sql_df(f"select * from DSA_{tab}", engine_source)
        
        # lire table dans ODS et drop la colonne load_date
        result_ods = sql_df(f"select * from ODS_{tab}", engine_target)
        
        if result_ods.empty:
            print(f"ODS_{tab} est vide, insert {result_dsa.shape[0]} ligne(s) dans la table ODS_{tab}")
            result_dsa.to_sql(name=f"ODS_{tab}", con=engine_target, if_exists='append', index=False)
        else:
            # surprimer colonne load_date
            result_ods.drop(columns='load_date', inplace=True)

            # comparer des donnees dans deux table ODS et DSA et prendre des donnees de DSA qui ne sont pas existes dans table ODS
            df_insert = result_dsa[~result_dsa.apply(tuple, axis=1).isin(result_ods.apply(tuple, axis=1))]

            # verifier si les donnees est existes
            if df_insert.empty:
                # insert rien comme les donnees n'existent pas
                print(f"Donnees est identique dans la table ODS_{tab}")
            else:
                # insert les donnees DSA qui n'existent pas dans ODS
                print(f"insert {df_insert.shape[0]} ligne(s) dans la table ODS_{tab}")
                df_insert.to_sql(name=f"ODS_{tab}", con=engine_target, if_exists='append', index=False)

    verifier_db_tab(tab_taget, engine_target)

    # fermer les connexions
    engine_source.dispose()
    engine_target.dispose()

flux_dsa_ods()

MOTEUR PRÊT ET VÉRIFIÉ : db_dsa
MOTEUR PRÊT ET VÉRIFIÉ : db_ods


,table_name,colonnes
0,DSA_Adresse,"id_adresse, adresse_numero, adresse_suffixe, a..."
1,DSA_Commune,"code_commune, nom_commune, ancien_code_commune..."
2,DSA_Culture,"code_nature_culture, nature_culture"
3,DSA_Culture_speciale,"code_nature_culture_speciale, nature_culture_s..."
4,DSA_Local,"code_type_local, type_local"
5,DSA_Lot,"id_lot, lot1_numero, lot1_surface_carrez, lot2..."
6,DSA_Mutation,"id_mutation, nature_mutation, date_mutation, v..."
7,DSA_Parcelle,"id_parcelle, ancien_id_parcelle, numero_dispos..."


,table_name
0,ODS_Adresse
1,ODS_Commune
2,ODS_Culture
3,ODS_Culture_speciale
4,ODS_Local
5,ODS_Lot
6,ODS_Mutation
7,ODS_Parcelle


,table_name,priority,colonnes
0,Commune,0,NaN
1,Local,1,NaN
2,Culture,2,NaN
3,Culture_speciale,3,NaN
4,Mutation,4,NaN
5,Parcelle,5,NaN
6,Adresse,6,NaN
7,Lot,7,NaN


Donnees est identique dans la table ODS_Commune
Donnees est identique dans la table ODS_Local
Donnees est identique dans la table ODS_Culture
Donnees est identique dans la table ODS_Culture_speciale
Donnees est identique dans la table ODS_Mutation
Donnees est identique dans la table ODS_Parcelle
Donnees est identique dans la table ODS_Adresse
Donnees est identique dans la table ODS_Lot
(44256, 8)


,id_adresse,adresse_numero,adresse_suffixe,adresse_code_voie,adresse_nom_voie,code_postal,code_commune,load_date
0,1,None,None,B038,LE HAMEL,27400,27339,2026-05-14 15:02:41
1,2,None,None,B046,LE SCABRE,27730,27119,2026-05-14 15:02:41
2,3,None,None,B026,LA MAISON ROUGE,27460,27008,2026-05-14 15:02:41
3,4,None,None,B030,LE VILLAGE,27500,27620,2026-05-14 15:02:41
4,5,None,None,B033,ROUVILLE,27460,27008,2026-05-14 15:02:41


... (10 outputs left out)

(47916, 5)


,id_mutation,nature_mutation,date_mutation,valeur_fonciere,load_date
0,2021-370814,Vente,2021-02-26,145000.0,2026-05-14 15:02:39
1,2021-372118,Vente,2021-04-07,30000.0,2026-05-14 15:02:39
2,2021-372840,Vente,2021-05-07,20000.0,2026-05-14 15:02:39
3,2021-372841,Vente,2021-05-06,2320.0,2026-05-14 15:02:39
4,2021-372842,Vente,2021-05-05,30000.0,2026-05-14 15:02:39


(75778, 7)


,id_parcelle,ancien_id_parcelle,numero_disposition,longitude,latitude,id_mutation,load_date
0,270010000A0016,None,1,0.715072,49.168240,2024-279960,2026-05-14 15:02:41
1,270010000A0061,None,1,0.710988,49.165466,2023-306894,2026-05-14 15:02:41
2,270010000A0078,None,1,0.711032,49.161631,2025-298261,2026-05-14 15:02:41
3,270010000A0079,None,1,0.710678,49.161290,2025-298261,2026-05-14 15:02:41
4,270010000A0094,None,1,0.708208,49.161878,2022-367198,2026-05-14 15:02:41


# ODS to DWH

### Charger table Dimension

In [80]:
# Charger table Dim_Date
def tad_date_dim(date_debut="2020-01-01", date_fin="2031-12-31"):
    # Creation un dataframe qui contient toutes les date entre "date_debut" et "date_fin"
    df_date = pd.date_range(start=date_debut, end=date_fin).to_frame(index=False, name='full_date')
    
    # Charger ce dataframe dans la table Dim_Date
    query = "select full_date from Dim_Date"
    result = sql_df(query, engine_db_dwh)
    
    if result.empty:
        print(f"Dim_Date est vide, insert {df_date.shape[0]} lignes dans la table")
        df_date.to_sql(name="Dim_Date", con=engine_db_dwh, if_exists='append', index=False)
    else:
        result['full_date'] = pd.to_datetime(result['full_date'])
        df_date_merge = df_date.merge(result, how='left', indicator=True)
        df_date_final = df_date_merge[df_date_merge['_merge']=='left_only'].drop(columns='_merge')
        if df_date_final.empty:
            print("Données déjà dans la table")
        else:
            print(f"Insert {df_date_final.shape[0]} lignes dans Dim_Date")
            df_date_final.to_sql(name="Dim_Date", con=engine_db_dwh, if_exists='append', index=False)

In [81]:
def sort_table(tab, key_table, key_sort='load_date'):
    df_sort = tab.sort_values(by=key_sort, ascending=False)
    return df_sort.drop(columns=key_sort).drop_duplicates(subset=key_table)

def tab_dim_dwh():
    # creer une dictionaire de cle primaire pour chaque table dans ODS
    list_pk_ods = {"ODS_Mutation":"id_mutation", "ODS_Commune":"code_commune", "ODS_Parcelle":"id_parcelle", "ODS_Local":"code_type_local", "ODS_Culture":"code_nature_culture", "ODS_Culture_speciale":"code_nature_culture_speciale", "ODS_Adresse":"id_adresse", "ODS_Lot":"id_lot"}
    list_npk_dwh = {"Dim_Localisation":"NPK_adresse", "Dim_Lot":"NPK_lot", "Dim_Mutation":"NPK_mutation", "Dim_Parcelle":"NPK_parcelle"}
    
    for tab, npk_tab in list_npk_dwh.items():
    # for tab in ["Dim_Localisation", "Dim_Lot", "Dim_Mutation", "Dim_Parcelle"]:
        tab_join = []
        if tab == "Dim_Localisation":
            tab_join = ["ODS_Adresse", "ODS_Commune"]
        elif tab == "Dim_Lot":
            tab_join = ["ODS_Lot", "ODS_Culture", "ODS_Culture_speciale", "ODS_Local"]
            
        if len(tab_join) > 0 and tab_join:
            df_default = sql_df(f"select * from {tab_join[0]}", engine_db_ods)
            df_default_sort = sort_table(df_default, list_pk_ods[tab_join[0]], key_sort='load_date')
            
            for i in range(len(tab_join)-1):
                df_join = sql_df(f"select * from {tab_join[i+1]}", engine_db_ods)
                df_join_sort = sort_table(df_join, list_pk_ods[tab_join[i+1]], key_sort='load_date')
                df_default_sort = df_default_sort.merge(df_join_sort, how='left')
        else:
            nom_table = tab.replace("Dim", "ODS")
            df_default = sql_df(f"select * from {nom_table}", engine_db_ods)
            df_default_sort = sort_table(df_default, list_pk_ods[nom_table], key_sort='load_date')

        # Đồng nhất tất cả null values thành NaN
        df_default_sort = df_default_sort.fillna(np.nan)
            
        result = sql_df(f"select * from {tab} where current_flag = 1", engine_db_dwh)
        result = result.fillna(np.nan)
        df_default_sort.columns = ['NPK_' + col[3:] if col.startswith('id_') else col for col in df_default_sort.columns]
        col_commun = [col for col in df_default_sort.columns if col in result.columns]
        df_default_col_commun = df_default_sort[col_commun]
        
        if result.empty:
            print(f"{tab} est vide, insert {df_default_col_commun.shape[0]} ligne(s) dans la table")
            df_default_col_commun.to_sql(name=tab, con=engine_db_dwh, if_exists='append', index=False)
        else:
            result_clean = result[col_commun]
            
            # comparer des donnees dans deux table ODS et DWH et prendre des donnees de ODS qui ne sont pas existes dans table DWH
            df_insert = df_default_col_commun[~df_default_col_commun.apply(tuple, axis=1).isin(result_clean.apply(tuple, axis=1))]

            # verifier si les donnees est existes
            if df_insert.empty:
                # insert rien comme les donnees n'existent pas
                print(f"Donnees sont deja dans {tab}")
            else:                
                # Tìm các giá trị trong df_insert có ở result_clean nhằm update với current_flag = 0 và date_fin = CURRENT_TIMESTAMP
                df_update = df_insert[df_insert[npk_tab].isin(result_clean[npk_tab])]
                
                #  chuyên thành dạng tuple
                print(f"update {df_update.shape[0]} ligne(s) dans la table {tab}")
                npk_tuple = tuple(df_update[npk_tab].to_list())  
                placeholders = ','.join([f"'{npk}'" for npk in npk_tuple])
                
                # update gia tri
                with engine_db_dwh.connect() as conn:
                    conn.execute(text(f"update {tab} set current_flag = 0, date_fin = CURRENT_TIMESTAMP where NPK_mutation in ({placeholders})"))
                    # commmit pour sauvegarder
                    conn.commit()
                
                # insert les donnees DSA qui n'existent pas dans ODS
                print(f"insert {df_insert.shape[0]} ligne(s) dans la table {tab}")
                # insert des donnees                    
                df_insert.to_sql(name=f"ODS_{tab}", con=engine_db_dwh, if_exists='append', index=False)

### Charger table Fait_DVF

In [85]:
# # read table ods
# tab_ods_mutation = sql_df("select distinct id_mutation as NPK_mutation, valeur_fonciere, date_mutation, load_date from ODS_Mutation order by load_date", engine_db_ods)
# tab_ods_lot = sql_df("select distinct id_parcelle as NPK_parcelle, id_lot as NPK_lot, id_adresse as NPK_adresse, load_date from ODS_Lot order by load_date", engine_db_ods)
# tab_ods_parcelle = sql_df("select distinct id_mutation as NPK_mutation, id_parcelle as NPK_parcelle, load_date from ODS_Parcelle order by load_date", engine_db_ods)
# tab_ods_adresse = sql_df("select distinct id_adresse as NPK_adresse, load_date from ODS_Adresse order by load_date", engine_db_ods)
# tab_ods_mutation.drop(columns="load_date", inplace=True) # (47916, 3)
# tab_ods_lot.drop(columns="load_date", inplace=True) # (121245, 2)
# tab_ods_parcelle.drop(columns="load_date", inplace=True) # (75778, 2)
# tab_ods_adresse.drop(columns="load_date", inplace=True) # (44256, 1)

# read table dwh
# tab_dim_date = sql_df("select distinct full_date as date_mutation, date_id from Dim_Date", engine_db_dwh)
# tab_dim_mutation = sql_df("select distinct NPK_mutation, TK_mutation, current_flag from Dim_Mutation where current_flag = 1", engine_db_dwh)
# tab_dim_lot = sql_df("select distinct NPK_lot, TK_lot, current_flag from Dim_Lot where current_flag = 1", engine_db_dwh)
# tab_dim_parcelle = sql_df("select distinct NPK_parcelle, TK_parcelle, current_flag from Dim_Parcelle where current_flag = 1", engine_db_dwh)
# tab_dim_localisation = sql_df("select distinct NPK_adresse, TK_adresse, current_flag from Dim_Localisation where current_flag = 1", engine_db_dwh)
# tab_dim_date : (4383, 2)
# tab_dim_mutation.drop(columns="current_flag", inplace=True) # (47916, 2)
# tab_dim_lot.drop(columns="current_flag", inplace=True) # (121355, 2)
# tab_dim_parcelle.drop(columns="current_flag", inplace=True) # (75778, 2)
# tab_dim_localisation.drop(columns="current_flag", inplace=True) # 44256, 2)

def read_ods_drop_date(list_col, nom_tab, key_table, engine_db, affiche=False):
    tab = sql_df(f"select distinct {list_col} from {nom_tab} order by load_date", engine_db)
    tab_sort = sort_table(tab, key_table, key_sort='load_date')
    
    if affiche:
        display(tab_sort.head())
        print(f"Dimension de {nom_tab} : {tab_sort.shape}")
    
    return tab_sort

def read_dwh_drop_date(list_col, nom_tab, engine_db, affiche=False):
    if nom_tab == "Dim_Date":
        tab = sql_df(f"select distinct {list_col} from {nom_tab}", engine_db)
    else:
        tab = sql_df(f"select distinct {list_col} from {nom_tab}  where current_flag = 1", engine_db)
        tab.drop(columns="current_flag", inplace=True)
    if affiche:
        display(tab.head())
        print(f"Dimension de {nom_tab} : {tab.shape}")
    
    return tab
    
def table_fait_dvf():
    dict_tab_colonne = {
        "ODS_Mutation" : {"col":"id_mutation as NPK_mutation, valeur_fonciere, date_mutation, load_date", "key":"NPK_mutation"},
        "ODS_Lot" : {"col":"id_lot as NPK_lot, id_parcelle as NPK_parcelle, id_adresse as NPK_adresse, load_date", "key":"NPK_lot"},
        "ODS_Parcelle" : {"col":"id_parcelle as NPK_parcelle, id_mutation as NPK_mutation, load_date", "key":"NPK_parcelle"},
        # "ODS_Adresse" : {"col":"id_adresse as NPK_adresse, load_date", "key":"NPK_adresse"},
        "Dim_Date" : {"col":"full_date as date_mutation, date_id", "key":"full_date"},
        "Dim_Mutation" : {"col":"NPK_mutation, TK_mutation, current_flag", "key":"NPK_mutation"},
        "Dim_Lot" : {"col":"NPK_lot, TK_lot, current_flag", "key":"NPK_lot"},
        "Dim_Parcelle" : {"col":"NPK_parcelle, TK_parcelle, current_flag", "key":"NPK_parcelle"},
        "Dim_Localisation" : {"col":"NPK_adresse, TK_adresse, current_flag", "key":"NPK_adresse"},
    }
    # read table ods
    tab_ods_mutation = read_ods_drop_date(dict_tab_colonne['ODS_Mutation']['col'], 'ODS_Mutation', dict_tab_colonne['ODS_Mutation']['key'], engine_db_ods)
    tab_ods_lot = read_ods_drop_date(dict_tab_colonne['ODS_Lot']['col'], 'ODS_Lot', dict_tab_colonne['ODS_Lot']['key'], engine_db_ods)
    tab_ods_parcelle = read_ods_drop_date(dict_tab_colonne['ODS_Parcelle']['col'], 'ODS_Parcelle', dict_tab_colonne['ODS_Parcelle']['key'], engine_db_ods)
    # tab_ods_adresse = read_ods_drop_date(dict_tab_colonne['ODS_Adresse']['col'], 'ODS_Adresse', dict_tab_colonne['ODS_Adresse']['key'], engine_db_ods)

    # read table dwh
    tab_dim_date = read_dwh_drop_date(dict_tab_colonne['Dim_Date']['col'], 'Dim_Date', engine_db_dwh)
    tab_dim_mutation = read_dwh_drop_date(dict_tab_colonne['Dim_Mutation']['col'], 'Dim_Mutation', engine_db_dwh)
    tab_dim_lot = read_dwh_drop_date(dict_tab_colonne['Dim_Lot']['col'], 'Dim_Lot', engine_db_dwh)
    tab_dim_parcelle = read_dwh_drop_date(dict_tab_colonne['Dim_Parcelle']['col'], 'Dim_Parcelle', engine_db_dwh)
    tab_dim_localisation = read_dwh_drop_date(dict_tab_colonne['Dim_Localisation']['col'], 'Dim_Localisation', engine_db_dwh)

    # jointure table
    # jointure des tables entre tab_ods_parcelle et tab_ods_lot pour creer df_dvf, (121355, 4)
    df_dvf = tab_ods_parcelle.merge(tab_ods_lot, how='inner', on='NPK_parcelle')
    # jointure des tables entre df_dvf et tab_dim_lot, (121355, 5)
    df_dvf = df_dvf.merge(tab_dim_lot, how='inner', on='NPK_lot')
    # jointure des tables entre df_dvf et tab_dim_localisation, (121355, 6)
    df_dvf = df_dvf.merge(tab_dim_localisation, how='left', on='NPK_adresse')
    # jointure des tables entre df_dvf et tab_dim_parcelle, (121355, 7)
    df_dvf = df_dvf.merge(tab_dim_parcelle, how='inner', on='NPK_parcelle')
    # jointure des tables entre df_dvf et tab_ods_mutation, (121355, 9)
    df_dvf = df_dvf.merge(tab_ods_mutation, how='inner', on='NPK_mutation')
    # jointure des tables entre df_dvf et tab_dim_mutation, (121355, 10) 
    df_dvf = df_dvf.merge(tab_dim_mutation, how='inner', on='NPK_mutation')
    # jointure finale des tables entre df_dvf et tab_dim_date, (121355, 11)
    df_dvf = df_dvf.merge(tab_dim_date, how='inner', on='date_mutation')
    print(df_dvf.info())

    # read donnees dans table de fait_dvf
    df_fait_dwh = sql_df(f"select * from Fait_DVF", engine_db_dwh)

    # clean et condition d'integration
    col_fait_dvf_test = tab_db_dwh.at[5, "colonnes"].split(',')
    col_dvf = [col.strip() for col in tab_db_dwh.at[5, "colonnes"].split(',')]
    dvf_final_charge = df_dvf[col_dvf]
    print(dvf_final_charge.info())
    
    if df_fait_dwh.empty:
        print(f"Fait_DVF est vide, insert {dvf_final_charge.shape[0]} ligne(s)")
        dvf_final_charge.to_sql(name='Fait_DVF', con=engine_db_dwh, if_exists='append', index=False)
    else:
        df_insert = dvf_final_charge[~dvf_final_charge.apply(tuple, axis=1).isin(df_fait_dwh.apply(tuple, axis=1))]
        if df_insert.empty:
            print("Donnees sont deja dans Fait_DVF")
        else:
            print(f"Insert {df_insert.shape[0]} ligne(s) dans la table Fait_DVF")
            df_insert.to_sql(name='Fait_DVF', con=engine_db_dwh, if_exists='append', index=False)


In [83]:
def flux_ods_dwh():
    # creation l'engine de 'db_ods'
    engine_db_ods = connect_engine_db('db_ods') 
    # creation l'engine de 'db_ods'
    engine_db_dwh = connect_engine_db('db_dwh')
    
    # creation la dataframe qui contient le nom de table et ses colonnes de base de donnees source
    tab_db_ods = sql_df_db_tab(engine_db_ods)
    # creation la dataframe qui contient le nom de table et ses colonnes de base de donnees target
    tab_db_dwh = sql_df_db_tab(engine_db_dwh)

    # flux pour chaque table
    # Dim_date
    tad_date_dim()
    # Autres tables dimensions
    tab_dim_dwh()
    # Table de fait Fait_DVF
    table_fait_dvf()  
    verifier_db_tab(tab_db_dwh, engine_db_dwh)

    # fermer les connexions
    engine_db_ods.dispose()
    engine_db_dwh.dispose()

In [84]:
flux_ods_dwh()

MOTEUR PRÊT ET VÉRIFIÉ : db_ods
MOTEUR PRÊT ET VÉRIFIÉ : db_dwh


,table_name,colonnes
0,ODS_Adresse,"id_adresse, adresse_numero, adresse_suffixe, a..."
1,ODS_Commune,"code_commune, nom_commune, ancien_code_commune..."
2,ODS_Culture,"code_nature_culture, nature_culture, load_date"
3,ODS_Culture_speciale,"code_nature_culture_speciale, nature_culture_s..."
4,ODS_Local,"code_type_local, type_local, load_date"
5,ODS_Lot,"id_lot, lot1_numero, lot1_surface_carrez, lot2..."
6,ODS_Mutation,"id_mutation, nature_mutation, date_mutation, v..."
7,ODS_Parcelle,"id_parcelle, ancien_id_parcelle, numero_dispos..."


,table_name,colonnes
0,Dim_Date,"date_id, full_date, jour, mois, trimestre, annee"
1,Dim_Localisation,"TK_adresse, NPK_adresse, adresse_numero, adres..."
2,Dim_Lot,"TK_lot, NPK_lot, lot1_numero, lot1_surface_car..."
3,Dim_Mutation,"TK_mutation, NPK_mutation, nature_mutation, da..."
4,Dim_Parcelle,"TK_parcelle, NPK_parcelle, ancien_id_parcelle,..."
5,Fait_DVF,"TK_lot, TK_adresse, TK_parcelle, TK_mutation, ..."


Données déjà dans la table
Donnees sont deja dans Dim_Localisation
Donnees sont deja dans Dim_Lot
Donnees sont deja dans Dim_Mutation
Donnees sont deja dans Dim_Parcelle
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121355 entries, 0 to 121354
Data columns (total 11 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   NPK_parcelle     121355 non-null  object 
 1   NPK_mutation     121355 non-null  object 
 2   NPK_lot          121355 non-null  int64  
 3   NPK_adresse      121355 non-null  int64  
 4   TK_lot           121355 non-null  int64  
 5   TK_adresse       121355 non-null  int64  
 6   TK_parcelle      121355 non-null  int64  
 7   valeur_fonciere  121060 non-null  float64
 8   date_mutation    121355 non-null  object 
 9   TK_mutation      121355 non-null  int64  
 10  date_id          121355 non-null  int64  
dtypes: float64(1), int64(7), object(3)
memory usage: 10.2+ MB
None
<class 'pandas.core.frame.DataFrame'>
Ra

,date_id,full_date,jour,mois,trimestre,annee
0,1,2020-01-01,1,1,1,2020
1,2,2020-01-02,2,1,1,2020
2,3,2020-01-03,3,1,1,2020
3,4,2020-01-04,4,1,1,2020
4,5,2020-01-05,5,1,1,2020


(44256, 15)


... (5 outputs left out)

(75778, 9)


,TK_parcelle,NPK_parcelle,ancien_id_parcelle,numero_diposition,longitude,latitude,date_debut,date_fin,current_flag
0,1,270010000A0016,None,None,0.715072,49.168240,2026-05-14 17:25:41,2099-12-31,1
1,2,274450000B0220,None,None,1.677006,49.244836,2026-05-14 17:25:41,2099-12-31,1
2,3,274450000B0207,None,None,1.678231,49.241650,2026-05-14 17:25:41,2099-12-31,1
3,4,274450000B0205,None,None,1.678310,49.241585,2026-05-14 17:25:41,2099-12-31,1
4,5,274450000B0203,None,None,1.676911,49.247475,2026-05-14 17:25:41,2099-12-31,1


(121355, 6)


,TK_lot,TK_adresse,TK_parcelle,TK_mutation,date_id,valeur_fonciere
0,1,1,13723,27887,749,245000.0
1,2,5345,34816,27304,387,30000.0
2,3,5354,14178,27169,402,275600.0
3,4,5354,14178,27169,402,275600.0
4,5,5354,14178,27169,402,275600.0


### Zone test

In [47]:
# # Tạo ODS đầy đủ hơn
# df_ods_full = pd.DataFrame({
#     'order_id': [1001, 1002, 1003, 1004, 1005],
#     'product': ['Laptop', 'Mouse', 'Keyboard', 'Monitor', 'USB'],
#     'quantity': [2, 5, 3, 1, 10],
#     'price': [1500, 25, 75, 300, 15],
#     'status': ['delivered', 'shipped', 'delivered', 'pending', 'shipped']
# })

# # Tạo df2 (dữ liệu mới cập nhật)
# df2 = pd.DataFrame({
#     'order_id': [1003, 1004, 1005, 1006, 1007, 1008],
#     'product': ['Keyboard', 'Monitor', 'USB', 'Webcam', 'Headset', 'Desk'],
#     'quantity': [3, 1, 12, 4, 2, 1],
#     'price': [75, 300, 15, 120, 85, 250],
#     'status': ['delivered', 'shipped', 'delivered', 'pending', 'shipped', 'pending']
# })

# print("=== df_ods_full ===")
# display(df_ods_full)
# print(f"\nShape: {df_ods_full.shape}")

# print("\n=== df2 ===")
# display(df2)
# print(f"\nShape: {df2.shape}")

In [48]:
# # Lọc các dòng chỉ có trong df2
# df_new = df2[~df2.apply(tuple, axis=1).isin(df_ods_full.apply(tuple, axis=1))]

# print("\n=== df_new (dòng mới chỉ có trong df2) ===")
# display(df_new)
# print(f"\nShape: {df_new.shape}")

In [49]:
# engine_db_test = connect_engine_db('db_test') 
# df_test_mutation = sql_df(f"select * from ODS_Mutation", engine_db_test)
# df_test_mutation_sort = df_test_mutation.sort_values(by='load_date', ascending=False)
# display(df_test_mutation_sort)
# display(df_test_mutation_sort.drop(columns='load_date').drop_duplicates(subset='id_mutation'))

In [50]:
# def sort_table(tab, key_table, key_sort='load_date'):
#     df_sort = tab.sort_values(by=key_sort, ascending=False)
#     return df_sort.drop(columns=key_sort).drop_duplicates(subset=key_table)

# # df_default = sql_df(f"select * from {nom_table}", engine_db_ods)
# # df_default_sort = sort_table(df_default, list_pk_ods[nom_table], key_sort='load_date')
# df_default_sort = pd.DataFrame({
#     'id_mutation': ['MUT001', 'MUT002', 'MUT003', 'MUT004', 'MUT015'],
#     'nature_mutation': ['Vente', 'Vente', 'Vente', 'Échange', 'Vente']
# })

# result = sql_df(f"select * from Dim_Mutation where current_flag = 1", engine_db_test)
# result = result.fillna(np.nan)
# display(result)
# df_default_sort.columns = ['NPK_' + col[3:] if col.startswith('id_') else col for col in df_default_sort.columns]

# col_commun = [col for col in df_default_sort.columns if col in result.columns]
# df_default_col_commun = df_default_sort[col_commun]
# display(df_default_col_commun)

# if result.empty:
#     print(f"{tab} est vide, insert {df_default_col_commun.shape[0]} ligne(s) dans la table")
#     df_default_col_commun.to_sql(name=tab, con=engine_db_dwh, if_exists='append', index=False)
# else:
#     result_clean = result[col_commun]
        
#     # comparer des donnees dans deux table ODS et DWH et prendre des donnees de ODS qui ne sont pas existes dans table DWH
#     df_insert = df_default_col_commun[~df_default_col_commun.apply(tuple, axis=1).isin(result_clean.apply(tuple, axis=1))]

#     # verifier si les donnees est existes
#     if df_insert.empty:
#         # insert rien comme les dnnees n'existent pas
#         print(f"Donnees sont deja dans {tab}")
#     else:
#         # insert les donnees DSA qui n'existent pas dans ODS
#         print(f"insert {df_insert.shape[0]} ligne(s) dans la table Dim_Mutation")
#         display(df_insert.head())
#         # update current_flag = 0, date_fin = current date (aujourd'hui) pour deactive des donnees
#         display(df_insert['NPK_mutation'])
#         # Tìm các giá trị trong df2 có ở df1
#         df_update = df_insert[df_insert['NPK_mutation'].isin(result_clean['NPK_mutation'])]

#         npk_tuple = tuple(df_update['NPK_mutation'].to_list())  # ('MUT001', 'MUT002', ...)
#         placeholders = ','.join([f"'{npk}'" for npk in npk_tuple])
        
#         with engine_db_test.connect() as conn:
#             conn.execute(text(f"update Dim_Mutation set current_flag = 0, date_fin = CURRENT_TIMESTAMP where NPK_mutation in ({placeholders})"))
#             # commmit pour sauvegarder
#             conn.commit()
                
#         # insert des donnees                    
#         df_insert.to_sql(name="Dim_Mutation", con=engine_db_test, if_exists='append', index=False)
                